# VocEd Lab 5 — Pixels as Vectors: k-NN and Logistic Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab5_pixels_as_vectors_knn_logistic_regression.ipynb)

Labs 2–4 squeezed every pixel down to **one** number — its grayscale brightness — and wrote the
rule by hand: *darker than `t_nucleus` means nucleus*.  This lab changes both halves of that:

- Each pixel keeps all **three** colour values, so it becomes a point in 3-D colour space.
- Instead of writing the rule, we let a **classifier** learn it from labelled training pixels.

We try two very different classifiers — **k-nearest neighbours**, which memorises, and
**logistic regression**, which fits a few weights — and ask whether colour beats Lab 4's
cleaned-up thresholds.

**Textbook:** [Chapter 5](https://obooks.tech/imageprocessing/chapter5.html) (Methods 1 and 2) and
[Appendix C](https://obooks.tech/imageprocessing/appendixc.html) (validation).

---

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

k-NN is slow to *predict*: some cells take a few minutes, and they say so.

## 0. Setup — Get the Data

The same images and the same train / validation / test split as Labs 3 and 4.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)

assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–4"
print("train:", len(train_idx), "  validation:", len(val_idx), "  test:", len(test_idx))

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

**The score to beat** — Lab 4's best thresholding pipeline (`specks, fill`): validation Dice
**0.812**, test Dice **0.772**.

## 1. A Pixel Is a Point in Colour Space

An image is stored as `(3, 256, 256)`: three colour planes.  To treat each pixel as one
**row** of three numbers (R, G, B), we move the colour axis to the end, then flatten the
256 × 256 grid into one long list of 65,536 pixels.

### 🧩 Puzzle 1 — From an image to a table of pixels

📖 Book: [§5.3 Visualising Colour Space](https://obooks.tech/imageprocessing/chapter5.html#visualising-colour-space)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

`.transpose(1, 2, 0)` reorders the axes from (channel, row, column) to (row, column, channel).

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

`.reshape(-1, 3)` means *three columns, and as many rows as needed*.

</details>

In [ ]:
pixels7 = X[7].transpose(___).reshape(___)     # (65536, 3)
labels7 = y[7].reshape(-1)                                # (65536,)
print("pixels:", pixels7.shape, "  labels:", labels7.shape)

rng = np.random.default_rng(0)
some = rng.choice(65536, 3000, replace=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(pixels7[some, 0], pixels7[some, 1], c=labels7[some], cmap='viridis', s=3)
axes[0].set_xlabel('red')
axes[0].set_ylabel('green')
axes[1].scatter(pixels7[some, 1], pixels7[some, 2], c=labels7[some], cmap='viridis', s=3)
axes[1].set_xlabel('green')
axes[1].set_ylabel('blue')
fig.suptitle('3,000 pixels of image 7 (purple = background, teal = cytoplasm, yellow = nucleus)')
plt.show()

# ── Check ──
assert pixels7.shape == (65536, 3), "One row per pixel, one column per colour"
assert np.allclose(pixels7[300], X[7][:, 1, 44]), "Pixel 300 is row 1, column 44 (300 = 256 + 44)"
print("✅ Puzzle 1 solved!")

### 🤔 Think 1 — Clouds in colour space

1. Which class forms the tightest cloud?  Why would you expect that class to be easy?
2. Which two classes overlap?  For a pixel in the overlap, what does any colour-only rule
   have to do?
3. Thresholding used only brightness — roughly, how far a point is along the diagonal from
   black to white.  Looking at the two plots, could knowing the *direction* away from the
   diagonal (the hue) help?

📖 Book: [§5.3 Visualising Colour Space](https://obooks.tech/imageprocessing/chapter5.html#visualising-colour-space)

✏️ **Your answer:** *(double-click this cell and type here)*

## 2. The Training Table

120 training images × 65,536 pixels is almost 8 million rows — far more than we need, and too
slow for k-NN.  We **sample** pixels from each training image.  There are two natural ways:

- **class-balanced:** 150 pixels of *each* class from every image, so the table has equal
  numbers of background, cytoplasm and nucleus;
- **random:** 450 pixels from anywhere in the image, so the classes keep their real shares.

First, what are the real shares?

In [ ]:
shares = np.bincount(y[train_idx].ravel()) / y[train_idx].size
print(f"Training pixels: background {shares[0]:.0%}, cytoplasm {shares[1]:.0%}, nucleus {shares[2]:.0%}")

### 🧩 Puzzle 2 — The class-balanced table

For every training image and every class, find the positions of that class's pixels and pick
150 of them at random.  (A few images have fewer than 150 nucleus pixels; `min` takes them all.)

📖 Book: [§5.4 Flattening Training Images into a Pixel Matrix](https://obooks.tech/imageprocessing/chapter5.html#flattening-training-images-into-a-pixel-matrix)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

`np.where(labels == c)[0]` gives the positions of the pixels whose label is `c`.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

`rng.choice(where, n, replace=False)` picks `n` different positions from `where`.

</details>

In [ ]:
rng = np.random.default_rng(0)
rows, row_labels = [], []
for i in train_idx:
    pixels = X[i].transpose(1, 2, 0).reshape(-1, 3)
    labels = y[i].reshape(-1)
    for c in [0, 1, 2]:
        where  = np.where(labels == ___)[0]
        chosen = rng.choice(where, min(150, len(where)), replace=False)
        rows.append(pixels[chosen])
        row_labels.append(labels[chosen])
X_bal, y_bal = np.vstack(rows), np.concatenate(row_labels)

# The random table, for comparison: 450 pixels from anywhere in each image
rng = np.random.default_rng(0)
rows, row_labels = [], []
for i in train_idx:
    chosen = rng.choice(65536, 450, replace=False)
    rows.append(X[i].transpose(1, 2, 0).reshape(-1, 3)[chosen])
    row_labels.append(y[i].reshape(-1)[chosen])
X_rand, y_rand = np.vstack(rows), np.concatenate(row_labels)

print("class-balanced table:", X_bal.shape, "  rows per class:", np.bincount(y_bal))
print("random table:        ", X_rand.shape, "  rows per class:", np.bincount(y_rand))

# ── Check ──
assert np.bincount(y_bal)[0] == 18000 and np.bincount(y_bal)[1] == 18000, "150 pixels of each class from each of 120 images"
print("✅ Puzzle 2 solved!")

## 3. k-Nearest Neighbours

`KNeighborsClassifier` stores the whole training table.  To label a new pixel it finds the `k`
training pixels with the closest colour and takes a majority vote.  `fit` only stores the table;
all the work happens in `predict`.

### 🧩 Puzzle 3 — Predict a whole mask

`predict` returns one label per row.  Turn those 65,536 labels back into a 256 × 256 mask.

📖 Book: [§5.5.3 Predicting Masks for Test Images](https://obooks.tech/imageprocessing/chapter5.html#predicting-masks-for-test-images)

<details><summary><b>💡 Hint</b> (click to open)</summary>

Puzzle 1 went from the image grid to a list; this goes back.  What shape is a mask?

</details>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def predict_mask(classifier, i):
    pixels = X[i].transpose(1, 2, 0).reshape(-1, 3)
    return classifier.predict(pixels).reshape(___)

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1).fit(X_bal, y_bal)
pred7 = predict_mask(knn, 7)
print(f"k-NN (k = 5) on image 7: Dice {image_dice(pred7, y[7]):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(y[7], cmap='viridis', vmin=0, vmax=2)
axes[0].set_title('true mask')
axes[1].imshow(pred7, cmap='viridis', vmin=0, vmax=2)
axes[1].set_title('k-NN, k = 5')
for ax in axes:
    ax.axis('off')
plt.show()

# ── Check ──
assert pred7.shape == (256, 256), "A mask is 256 × 256"
print("✅ Puzzle 3 solved!")

### Choosing k on the validation images

`k` is k-NN's knob.  We choose it the way Lab 3 taught: try several values, score each on
**validation** images, keep the best.  Predicting is slow, so we use the first **10** validation
images.  We also print each model's accuracy on its own training table.  *(About 3 minutes.)*

### 🧩 Puzzle 4 — Pick the best k

📖 Book: [§5.5.2 Choosing k](https://obooks.tech/imageprocessing/chapter5.html#choosing-k)

<details><summary><b>💡 Hint</b> (click to open)</summary>

You need the *position* of the largest validation Dice — the same NumPy function as in Lab 3, Puzzle 2.

</details>

In [ ]:
val10 = val_idx[:10]
k_values = [1, 5, 25, 75]
val_k = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1).fit(X_bal, y_bal)
    val_k.append(np.mean([image_dice(predict_mask(knn, i), y[i]) for i in val10]))
    print(f"k = {k:3d}   training accuracy {knn.score(X_bal, y_bal):.3f}   validation Dice {val_k[-1]:.4f}")

best_k = k_values[np.___(val_k)]
print("Best k:", best_k)

# ── Check ──
assert best_k == k_values[int(np.argmax(val_k))], "Take the k with the largest validation Dice"
print("✅ Puzzle 4 solved!")

### 🤔 Think 2 — k = 1 remembers everything

1. With `k = 1`, training accuracy is almost perfect.  Why?  *(Which training pixel is
   closest to a training pixel?)*
2. Yet `k = 1` has the **worst** validation Dice.  What is it doing wrong in the overlap of
   the cytoplasm and nucleus clouds?
3. Which way does training accuracy move as `k` grows, and which way does validation Dice move?
   Why would you never choose `k` by training accuracy?

📖 Book: [Appendix C — Overfitting and Underfitting](https://obooks.tech/imageprocessing/appendixc.html#overfitting-and-underfitting)

✏️ **Your answer:** *(double-click this cell and type here)*

### Balanced or random?

Now the other table, with the best `k`, on the same 10 validation images.  *(About 45 seconds.)*

In [ ]:
knn_bal  = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1).fit(X_bal,  y_bal)
knn_rand = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1).fit(X_rand, y_rand)

for name, model in [('class-balanced', knn_bal), ('random', knn_rand)]:
    preds = np.stack([predict_mask(model, i) for i in val10])
    truth = y[val10]
    dice  = np.mean([image_dice(p, y[i]) for p, i in zip(preds, val10)])
    nuc_dice = np.mean([dice_score(p, y[i], 2) for p, i in zip(preds, val10)])
    cyto_as_nuc = (preds[truth == 1] == 2).mean()
    print(f"{name:15s} validation Dice {dice:.4f}   nucleus Dice {nuc_dice:.4f}   "
          f"true cytoplasm called nucleus {cyto_as_nuc:.1%}")

### 🤔 Think 3 — Balanced is not always better

Lab 2 taught us to *stratify* the train / test split, so it is tempting to think that a
class-balanced training table must be better too.

1. Which table gave the better validation Dice?
2. In the balanced table, a third of the rows are nucleus and a third cytoplasm.  In the real
   images, what are the shares (printed in Section 2)?  Which class is over-represented in the
   balanced table, and which under-represented?
3. For a pixel in the cytoplasm–nucleus overlap, the vote is decided by how many neighbours of
   each class are nearby.  Explain how the balanced table tips that vote — and find the line in the
   output above that shows it happening.
4. Stratifying a train / test split and balancing a training table are different things.  What
   is each one for?

📖 Book: [Appendix C — Stratification](https://obooks.tech/imageprocessing/appendixc.html#stratification)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. Logistic Regression

Logistic regression learns **three weights per class** (one for each colour) plus a constant.
For a pixel it computes a score for each class — `w_R·R + w_G·G + w_B·B + b` — and picks the
highest.  The boundary between two classes is a flat plane in colour space.  It trains in a
second and predicts very fast.  We train it on the random table.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000).fit(X_rand, y_rand)
val_lr = np.mean([image_dice(predict_mask(lr, i), y[i]) for i in val_idx])
print(f"Logistic regression, all 40 validation images: Dice {val_lr:.4f}")

print()
print("Mean colour:     red     green    blue")
for c, name in enumerate(['background', 'cytoplasm', 'nucleus']):
    m = np.concatenate([X[i][:, y[i] == c] for i in train_idx], axis=1).mean(axis=1)
    print(f"  {name:11s} {m[0]:7.2f}  {m[1]:7.2f}  {m[2]:7.2f}")

print()
print("Weights:        red     green    blue    constant")
for c, name in enumerate(['background', 'cytoplasm', 'nucleus']):
    w = lr.coef_[c]
    print(f"  {name:11s} {w[0]:7.2f}  {w[1]:7.2f}  {w[2]:7.2f}   {lr.intercept_[c]:7.2f}")

### 🧩 Puzzle 5 — Which colour says "nucleus"?

`lr.coef_[2]` holds the nucleus class's three weights (red, green, blue).  A large **negative**
weight means *the more of this colour, the less likely nucleus*.  Find the colour with the most
negative nucleus weight.

📖 Book: [§5.6 Method 2 — Logistic Regression](https://obooks.tech/imageprocessing/chapter5.html#method-2-logistic-regression)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`np.argmin` gives the position of the smallest value.

</details>

In [ ]:
colours = ['red', 'green', 'blue']
strongest = colours[np.___(lr.coef_[2])]
print("The colour whose absence most says 'nucleus':", strongest)

# ── Check ──
assert strongest == colours[int(np.argmin(lr.coef_[2]))], "Take the most negative of the three nucleus weights"
print("✅ Puzzle 5 solved!")

### 🤔 Think 4 — Reading the weights

1. Which colour is the nucleus's strongest sign?  Look at the mean colours: going from
   cytoplasm to nucleus, which channel drops the most?
2. Grayscale weights green most (`0.587·G`).  Was that a lucky choice for Labs 2–4?
3. Logistic regression's boundaries are flat planes.  Compare its validation Dice with k-NN's,
   which can draw any shape.  What does that suggest about the shape of the boundary the data
   needs?

📖 Book: [§5.6 Method 2 — Logistic Regression](https://obooks.tech/imageprocessing/chapter5.html#method-2-logistic-regression)

✏️ **Your answer:** *(double-click this cell and type here)*

## 5. The Final Test — Once

We take the validation winners — k-NN with the best `k` on the random table, and logistic
regression — to the 40 test images.  *(About 3 minutes: k-NN is predicting 2.6 million pixels.)*

📖 Book: [§5.9 Comparing the Methods](https://obooks.tech/imageprocessing/chapter5.html#comparing-the-methods)

In [ ]:
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

print(f"{'':28s} test Dice   N/C R²   N/C within ±0.05")
for name, model in [(f'k-NN (k = {best_k}, random table)', knn_rand), ('logistic regression', lr)]:
    preds   = [predict_mask(model, i) for i in test_idx]
    dice    = np.mean([image_dice(p, y[i]) for p, i in zip(preds, test_idx)])
    nc_pred = np.array([nc_ratio(p) for p in preds])
    r2      = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
    close   = np.mean(np.abs(nc_pred - nc_true) < 0.05)
    print(f"{name:28s} {dice:.4f}    {r2:7.3f}   {close:.0%}")

print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 2 (0.40, 0.85, blur σ=2)    0.719")
print("  Lab 3 five-knob pipeline        0.770     -0.812")
print("  Lab 4 specks, fill              0.772     -0.465   35%")

### 🤔 Think 5 — Did colour win?

1. Compare k-NN and logistic regression with Lab 4.  Did three colour values and a learned rule
   beat one brightness value and a hand-written rule with clean-up?
2. Lab 4 also **cleaned** its masks (specks, holes).  The classifiers here label every pixel on
   its own, with no clean-up.  Is this a fair comparison?  What would you try next to make it fair?
3. A classifier that only sees one pixel's colour has a ceiling: in the overlap of two clouds,
   no rule can be right every time.  What *other* information could tell two pixels of the same
   colour apart?

📖 Book: [§5.10 Why All Colour-Only Methods Hit a Ceiling](https://obooks.tech/imageprocessing/chapter5.html#why-all-colour-only-methods-hit-a-ceiling)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **A pixel is a point in colour space.**  `transpose(1, 2, 0).reshape(-1, 3)` turns an image
  into a table with one row per pixel; `.reshape(256, 256)` turns predictions back into a mask.
- **k-NN memorises; logistic regression fits weights.**  k-NN can draw any boundary but is slow
  to predict; logistic regression draws flat planes and is fast.
- **Choose `k` on validation, never on training accuracy.**  `k = 1` is perfect on its training
  table and worst on new images.
- **Balancing the training table is not free.**  It changes what the model believes about how
  common each class is.  Here the random table was better.
- **Green is the nucleus channel.**  The weights of a simple model can teach you about the data.
- **Colour alone ≈ Lab 4.**  One pixel's colour has a ceiling — Lab 6 measures it.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run Sections 3–4 |
|---|---|
| A | `k_values = [10, 25, 50, 150]` |
| B | 150 random pixels per image instead of 450 (a smaller table) |
| C | `LogisticRegression(max_iter=2000, class_weight='balanced')` on the random table |

1. How much does the best `k` matter, once it is not tiny?
2. Does a smaller table cost much Dice?  What does it save?
3. `class_weight='balanced'` is another way to balance classes.  Does it help or hurt, and why —
   given what you found in Think 3?